# 👤 LCMSeg_turkm — Human Evaluation

**Два этапа:**

**Этап 1** (ячейки H1–H3) — подготовка CSV для эксперта.
Запускается один раз, файл отправляется лингвисту.

**Этап 2** (ячейки H4–H5) — подсчёт метрик после заполнения CSV.
Запускается после получения заполненного файла от эксперта.

**Инструкция для эксперта:**
Откройте файл `human_eval_words.csv` и заполните три колонки:
- `correct` → **TRUE** (верно) / **FALSE** (неверно) / **PARTIAL** (частично)
- `gold_seg` → правильная сегментация через `@@ ` (только если FALSE/PARTIAL)
- `comment` → необязательный комментарий

**Пример заполнения:**
```
word         femseg_seg          correct   gold_seg            comment
kitaplar     kitap@@ lar         TRUE
häzirki      h@@ äzirki          FALSE     häzirki             ошибочное дробление
adamlaryň    adam@@ lar@@ yň     TRUE
geldim       gel@@ dim           TRUE
```

## 📦 Ячейка H1 — Установка и загрузка модели

In [ ]:
!pip install -q pytorch-crf
from google.colab import drive
import os, re, json, csv, torch, torch.nn as nn
from torchcrf import CRF
from tqdm import tqdm
drive.mount('/content/drive')

MODEL_BASE  = '/content/drive/MyDrive/TURKM_MORPH'
CORPUS_NAME = 'turkmen_80000_segmented'
MODEL_DIR   = os.path.join(MODEL_BASE, f'models_{CORPUS_NAME}')
BEST_PT     = os.path.join(MODEL_DIR, f'lcmseg_turkm_{CORPUS_NAME}_BEST.pt')

FRONT_VOWELS=set('eiöü'); BACK_VOWELS=set('ayou')
HARMONY_FRONT=0; HARMONY_BACK=1; HARMONY_CONSONANT=2

def norm_turkm(s):
    return str(s).replace('\u0148','\u00f1').replace('\u0147','\u00d1').replace('\ufeff','').strip()
def char_harmony_class(ch):
    c=ch.lower()
    if c in FRONT_VOWELS: return HARMONY_FRONT
    if c in BACK_VOWELS: return HARMONY_BACK
    return HARMONY_CONSONANT
def word_harmony_class(word):
    for ch in reversed(word.lower()):
        cls=char_harmony_class(ch)
        if cls!=HARMONY_CONSONANT: return cls
    return HARMONY_CONSONANT
def is_vowel(ch): return 1 if ch.lower() in (FRONT_VOWELS|BACK_VOWELS) else 0

TAGS=['B','M','E','S']; TAG2ID={t:i for i,t in enumerate(TAGS)}
ID2TAG={i:t for t,i in TAG2ID.items()}

class TurkmLCMSegCRF(nn.Module):
    def __init__(self,vocab_size,tagset_size,emb_dim=128,harmony_dim=16,
                 vowel_dim=8,word_harm_dim=16,hidden_dim=256,dropout=0.3):
        super().__init__()
        self.char_emb=nn.Embedding(vocab_size,emb_dim,padding_idx=0)
        self.harmony_emb=nn.Embedding(3,harmony_dim)
        self.is_vowel_emb=nn.Embedding(2,vowel_dim)
        self.word_harm_emb=nn.Embedding(3,word_harm_dim)
        in_dim=emb_dim+harmony_dim+vowel_dim+word_harm_dim
        self.lstm=nn.LSTM(in_dim,hidden_dim,num_layers=2,bidirectional=True,
                          batch_first=True,dropout=dropout)
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(hidden_dim*2,tagset_size)
        self.crf=CRF(tagset_size,batch_first=True)
    def forward(self,inp,harm,vow,wh,tags=None,mask=None):
        x=torch.cat([self.char_emb(inp),self.harmony_emb(harm),
                      self.is_vowel_emb(vow),self.word_harm_emb(wh)],dim=-1)
        x,_=self.lstm(x); x=self.dropout(x); emis=self.fc(x)
        if tags is not None: return -self.crf(emis,tags,mask=mask,reduction='mean')
        return self.crf.decode(emis,mask=mask)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt=torch.load(BEST_PT,map_location=device)
char2id=ckpt['char2id']
femseg=TurkmLCMSegCRF(vocab_size=len(char2id),tagset_size=4).to(device)
femseg.load_state_dict(ckpt['state_dict']); femseg.eval()
print(f'✅ LCMSeg загружен: morph_acc={ckpt["val_morph"]:.4f}')


Mounted at /content/drive
✅ LCMSeg загружен: morph_acc=0.9941


In [ ]:
import re, random, csv, os, glob
random.seed(42)

MODEL_BASE = '/content/drive/MyDrive/TURKM_MORPH'
SEG_DIR    = '/content/drive/MyDrive/SEGMENTATION_TURKM/turkmen_80000_segmented'
csv_path   = os.path.join(MODEL_BASE, 'human_eval_words.csv')

def norm_turkm(s):
    return str(s).replace('\u0148','\u00f1').replace('\u0147','\u00d1').replace('\ufeff','').strip()

# Читаем слова из корпуса (первые 5 чанков — достаточно)
pat = re.compile(r'[a-zA-ZçñöşüýžÇÑÖŞÜÝŽ]+')
all_words = set()

chunk_files = sorted(glob.glob(os.path.join(SEG_DIR, 'chunk_*.txt')))[:5]
print(f'Читаем {len(chunk_files)} чанков...')

for chunk in chunk_files:
    with open(chunk, encoding='utf-8') as f:
        for line in f:
            # Убираем @@ и берём слова
            clean = line.replace('@@ ', '').replace('@@', '')
            tokens = pat.findall(norm_turkm(clean))
            all_words.update(t.lower() for t in tokens if len(t) > 3)

print(f'Уникальных слов: {len(all_words)}')

# Перемешиваем случайно
words_list = list(all_words)
random.shuffle(words_list)

# Стратифицированная выборка
short = [w for w in words_list if 3 < len(w) <= 5][:100]
mid   = [w for w in words_list if 5 < len(w) <= 9][:100]
long  = [w for w in words_list if len(w) >= 10][:100]
eval_words = short + mid + long

from collections import Counter
letters = Counter(w[0] for w in eval_words)
print(f'Букв разных: {len(letters)}')
print(f'Распределение: {dict(sorted(letters.items()))}')

# Записываем БЕЗ сегментации (эксперт увидит только слова)
if os.path.exists(csv_path):
    os.remove(csv_path)

rows = []
for word in eval_words:
    rows.append({
        'word':        word,
        'femseg_seg':  '',   # заполним отдельно если нужно
        'n_morphemes': '',
        'correct':     '',
        'gold_seg':    '',
        'comment':     '',
    })

with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f,
        fieldnames=['word','femseg_seg','n_morphemes','correct','gold_seg','comment'])
    writer.writeheader()
    writer.writerows(rows)

print(f'✅ Записано {len(rows)} слов')
print(f'Первые 10: {[r["word"] for r in rows[:10]]}')

Читаем 5 чанков...
Уникальных слов: 33992
Букв разных: 30
Распределение: {'a': 22, 'b': 28, 'c': 1, 'd': 7, 'e': 13, 'f': 5, 'g': 31, 'h': 7, 'i': 9, 'j': 5, 'k': 15, 'l': 6, 'm': 21, 'n': 6, 'o': 6, 'p': 11, 'r': 8, 's': 20, 't': 20, 'u': 3, 'v': 1, 'w': 7, 'x': 1, 'y': 5, 'z': 2, 'ç': 7, 'ö': 6, 'ü': 1, 'ý': 20, 'ş': 6}
✅ Записано 300 слов
Первые 10: ['boldy', 'xvii', 'tenis', 'ýetip', 'ýogy', 'tanka', 'tumen', 'seýan', 'döwri', 'aýman']


In [ ]:
import re, random, csv, os, glob, torch, torch.nn as nn, json
random.seed(42)

MODEL_BASE  = '/content/drive/MyDrive/TURKM_MORPH'
CORPUS_NAME = 'turkmen_80000_segmented'
MODEL_DIR   = os.path.join(MODEL_BASE, f'models_{CORPUS_NAME}')
BEST_PT     = os.path.join(MODEL_DIR, f'lcmseg_turkm_{CORPUS_NAME}_BEST.pt')
csv_path    = os.path.join(MODEL_BASE, 'human_eval_words.csv')

# Читаем слова из CSV
with open(csv_path, encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
words = [r['word'] for r in rows]
print(f'Слов для сегментации: {len(words)}')

# Загружаем модель (если не в памяти)
from torchcrf import CRF
FRONT_VOWELS=set('eiöü'); BACK_VOWELS=set('ayou')
HARMONY_FRONT=0; HARMONY_BACK=1; HARMONY_CONSONANT=2

def norm_turkm(s):
    return str(s).replace('\u0148','\u00f1').replace('\u0147','\u00d1').replace('\ufeff','').strip()
def char_harmony_class(ch):
    c=ch.lower()
    if c in FRONT_VOWELS: return HARMONY_FRONT
    if c in BACK_VOWELS: return HARMONY_BACK
    return HARMONY_CONSONANT
def word_harmony_class(word):
    for ch in reversed(word.lower()):
        cls=char_harmony_class(ch)
        if cls!=HARMONY_CONSONANT: return cls
    return HARMONY_CONSONANT
def is_vowel(ch): return 1 if ch.lower() in (FRONT_VOWELS|BACK_VOWELS) else 0

TAGS=['B','M','E','S']; TAG2ID={t:i for i,t in enumerate(TAGS)}
ID2TAG={i:t for t,i in TAG2ID.items()}

class TurkmFEMSegCRF(nn.Module):
    def __init__(self,vocab_size,tagset_size,emb_dim=128,harmony_dim=16,
                 vowel_dim=8,word_harm_dim=16,hidden_dim=256,dropout=0.3):
        super().__init__()
        self.char_emb=nn.Embedding(vocab_size,emb_dim,padding_idx=0)
        self.harmony_emb=nn.Embedding(3,harmony_dim)
        self.is_vowel_emb=nn.Embedding(2,vowel_dim)
        self.word_harm_emb=nn.Embedding(3,word_harm_dim)
        in_dim=emb_dim+harmony_dim+vowel_dim+word_harm_dim
        self.lstm=nn.LSTM(in_dim,hidden_dim,num_layers=2,bidirectional=True,
                          batch_first=True,dropout=dropout)
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(hidden_dim*2,tagset_size)
        self.crf=CRF(tagset_size,batch_first=True)
    def forward(self,inp,harm,vow,wh,tags=None,mask=None):
        x=torch.cat([self.char_emb(inp),self.harmony_emb(harm),
                      self.is_vowel_emb(vow),self.word_harm_emb(wh)],dim=-1)
        x,_=self.lstm(x); x=self.dropout(x); emis=self.fc(x)
        if tags is not None: return -self.crf(emis,tags,mask=mask,reduction='mean')
        return self.crf.decode(emis,mask=mask)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt=torch.load(BEST_PT,map_location=device)
char2id=ckpt['char2id']
femseg=TurkmFEMSegCRF(vocab_size=len(char2id),tagset_size=4).to(device)
femseg.load_state_dict(ckpt['state_dict']); femseg.eval()
print(f'✅ Модель загружена: morph_acc={ckpt["val_morph"]:.4f}')

# Сегментируем
VALID_SHORT={'da','de','dan','den','a','e','ny','ni','dy','di',
             'lar','ler','yň','iň','sy','si','ly','li','na','ne'}
MERGE_RIGHT={'in','un','ün','yn'}
NO_SPLIT={'türkmen','türkmenistan','döwlet','garaşsyz','mekdep','taýýar'}

def seg_word(word):
    w=norm_turkm(word)
    w_l=w.lower()
    # NO_SPLIT проверка
    for stem in sorted(NO_SPLIT,key=len,reverse=True):
        if w_l.startswith(stem):
            suf=w[len(stem):]
            if not suf: return [w]
            break
    chars=list(w)
    if not chars: return [word]
    def t(lst,dt): return torch.tensor([lst],dtype=dt).to(device)
    inp=t([char2id.get(ch,char2id['<unk>']) for ch in chars],torch.long)
    harm=t([char_harmony_class(ch) for ch in chars],torch.long)
    vow=t([is_vowel(ch) for ch in chars],torch.long)
    wh=t([word_harmony_class(w)]*len(chars),torch.long)
    mask=torch.ones(1,len(chars),dtype=torch.bool).to(device)
    with torch.no_grad(): paths=femseg(inp,harm,vow,wh,tags=None,mask=mask)
    pred=[ID2TAG[tt] for tt in paths[0]]
    morphs,buf=[],[]
    for ch,tag in zip(chars,pred):
        buf.append(ch)
        if tag in('E','S'): morphs.append(''.join(buf)); buf=[]
    if buf: morphs.append(''.join(buf))
    # Постобработка
    if len(morphs)<=1: return morphs
    merged,i=[],0
    while i<len(morphs):
        m=morphs[i]
        if m.lower() in MERGE_RIGHT and i+1<len(morphs):
            merged.append(m+morphs[i+1]); i+=2
        else: merged.append(m); i+=1
    result=[merged[0]]
    for m in merged[1:]:
        if m in('ñ','ň'): result[-1]+=m
        elif len(m)==1 and m.lower() not in VALID_SHORT: result[-1]+=m
        else: result.append(m)
    return result

print('Сегментируем...')
new_rows = []
for r in rows:
    word = r['word']
    morphs = seg_word(word)
    seg_str = '@@ '.join(morphs) if len(morphs)>1 else morphs[0]
    new_rows.append({
        'word':        word,
        'femseg_seg':  seg_str,
        'n_morphemes': len(morphs),
        'correct':     '',
        'gold_seg':    '',
        'comment':     '',
    })

# Перезаписываем
os.remove(csv_path)
with open(csv_path,'w',encoding='utf-8-sig',newline='') as f:
    writer=csv.DictWriter(f,
        fieldnames=['word','femseg_seg','n_morphemes','correct','gold_seg','comment'])
    writer.writeheader(); writer.writerows(new_rows)

print(f'✅ CSV перезаписан с сегментацией: {len(new_rows)} слов')
print(f'\nПримеры:')
for r in new_rows[:10]:
    print(f"  {r['word']:<25} → {r['femseg_seg']}")

Слов для сегментации: 300
✅ Модель загружена: morph_acc=0.9941
Сегментируем...
✅ CSV перезаписан с сегментацией: 300 слов

Примеры:
  boldy                     → bol@@ dy
  xvii                      → xvii
  tenis                     → tenis
  ýetip                     → ýet@@ ip
  ýogy                      → ýogy
  tanka                     → tanka
  tumen                     → tumen
  seýan                     → seýan
  döwri                     → döwri
  aýman                     → aým@@ an


## ✂️ Ячейка H2 — Сегментация и подготовка CSV для эксперта

In [ ]:
NO_SPLIT_STEMS={'türkmen','türkmenistan','döwlet','garaşsyz','mekdep','taýýar'}
VALID_SHORT={'da','de','dan','den','a','e','ny','ni','dy','di',
             'lar','ler','yň','iň','sy','si','ly','li','na','ne'}
MERGE_RIGHT={'in','un','ün','yn'}

def check_no_split(word):
    w=word.lower()
    if w in NO_SPLIT_STEMS: return True,word,''
    for stem in sorted(NO_SPLIT_STEMS,key=len,reverse=True):
        if w.startswith(stem) and len(word)>len(stem):
            return True,word[:len(stem)],word[len(stem):]
    return False,word,''

def postprocess(morphs):
    if len(morphs)<=1: return morphs
    merged,i=[],0
    while i<len(morphs):
        m=morphs[i]
        if m.lower() in MERGE_RIGHT and i+1<len(morphs):
            merged.append(m+morphs[i+1]); i+=2
        else: merged.append(m); i+=1
    result=[merged[0]]
    for m in merged[1:]:
        if m in('ñ','ň','Ñ','Ň'): result[-1]+=m
        elif len(m)==1 and m.lower() not in VALID_SHORT: result[-1]+=m
        else: result.append(m)
    return result

def seg_word(word,model,c2id,dev):
    w=norm_turkm(word)
    hit,stem,suf=check_no_split(w)
    if hit:
        if not suf: return [stem]
        chars=list(suf)
        if not chars: return [stem]
        def t(lst,dt): return torch.tensor([lst],dtype=dt).to(dev)
        inp=t([c2id.get(ch,c2id['<unk>']) for ch in chars],torch.long)
        harm=t([char_harmony_class(ch) for ch in chars],torch.long)
        vow=t([is_vowel(ch) for ch in chars],torch.long)
        wh=t([word_harmony_class(suf)]*len(chars),torch.long)
        mask=torch.ones(1,len(chars),dtype=torch.bool).to(dev)
        with torch.no_grad(): paths=model(inp,harm,vow,wh,tags=None,mask=mask)
        pred=[ID2TAG[tt] for tt in paths[0]]
        morphs,buf=[],[]
        for ch,tag in zip(chars,pred):
            buf.append(ch)
            if tag in('E','S'): morphs.append(''.join(buf)); buf=[]
        if buf: morphs.append(''.join(buf))
        return [stem]+postprocess(morphs)
    chars=list(w)
    if not chars: return [word]
    def t(lst,dt): return torch.tensor([lst],dtype=dt).to(dev)
    inp=t([c2id.get(ch,c2id['<unk>']) for ch in chars],torch.long)
    harm=t([char_harmony_class(ch) for ch in chars],torch.long)
    vow=t([is_vowel(ch) for ch in chars],torch.long)
    wh=t([word_harmony_class(w)]*len(chars),torch.long)
    mask=torch.ones(1,len(chars),dtype=torch.bool).to(dev)
    with torch.no_grad(): paths=model(inp,harm,vow,wh,tags=None,mask=mask)
    pred=[ID2TAG[tt] for tt in paths[0]]
    morphs,buf=[],[]
    for ch,tag in zip(chars,pred):
        buf.append(ch)
        if tag in('E','S'): morphs.append(''.join(buf)); buf=[]
    if buf: morphs.append(''.join(buf))
    return postprocess(morphs)

# =============================================================
# Загружаем туркменские предложения FLORES-200
# Вставьте сюда предложения из ячейки E3 или загрузите снова
# =============================================================
import glob
# CSV уже готов — просто читаем его напрямую
csv_ready = os.path.join(MODEL_BASE, 'human_eval_words.csv')
print(f'✅ CSV уже существует: {csv_ready}')
print(f'   Отправьте его эксперту-лингвисту для заполнения.')
print(f'   После получения ответа запускайте Ячейку H4.')

import os
csv_path = os.path.join(MODEL_BASE, 'human_eval_words.csv')

# Принудительно удаляем и пересоздаём
if os.path.exists(csv_path):
    os.remove(csv_path)

with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f,
        fieldnames=['word','femseg_seg','n_morphemes','correct','gold_seg','comment'])
    writer.writeheader()
    writer.writerows(rows)

print(f'✅ Файл перезаписан: {csv_path}')
print(f'   Слов: {len(rows)}')

# Проверяем первые буквы
from collections import Counter
letters = Counter(r['word'][0] for r in rows)
print(f'   Первые буквы: {dict(sorted(letters.items()))}')
# Показываем первые строки для проверки
with open(csv_ready, encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    rows = list(reader)
print(f'\n   Слов в файле: {len(rows)}')
print(f'\n   Первые 5 строк:')
# Восстанавливаем flores_sentences из CSV для дальнейших ячеек
# (не нужны — CSV уже готов, пропускаем остаток ячейки)
import sys
print('\n✅ CSV готов. Переходите к Ячейке H4.')
sys.exit(0)
for r in rows[:5]:
    print(f"   {r['word']:<20} → {r['femseg_seg']}")

# Извлекаем слова
pat = re.compile(r'[a-zA-ZçñöşüýžÇÑÖŞÜÝŽ]+')
all_words_raw = []
for sent in flores_sentences:
    all_words_raw.extend(pat.findall(norm_turkm(sent)))
unique_words = list(set(w.lower() for w in all_words_raw if len(w) > 3))

print(f'Уникальных слов (>3 букв): {len(unique_words)}')

# Сегментируем
dev_str = str(device)
segmented = {}
for word in tqdm(unique_words, desc='Сегментация'):
    segmented[word] = seg_word(word, femseg, char2id, dev_str)

# Стратифицированная выборка для эксперта
import random
random.seed(42)

short_pool = [w for w in unique_words if 3 < len(w) <= 5]
mid_pool   = [w for w in unique_words if 5 < len(w) <= 9]
long_pool  = [w for w in unique_words if len(w) >= 10]

short_w = random.sample(short_pool, min(100, len(short_pool)))
mid_w   = random.sample(mid_pool,   min(100, len(mid_pool)))
long_w  = random.sample(long_pool,  min(100, len(long_pool)))
eval_words = short_w + mid_w + long_w

rows = []
for word in eval_words:
    morphs  = segmented.get(word, [word])
    seg_str = '@@ '.join(morphs) if len(morphs)>1 else morphs[0]
    rows.append({
        'word':        word,
        'femseg_seg':  seg_str,
        'n_morphemes': len(morphs),
        'correct':     '',
        'gold_seg':    '',
        'comment':     '',
    })

csv_path = os.path.join(MODEL_BASE, 'human_eval_words.csv')
with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
    writer=csv.DictWriter(f,
        fieldnames=['word','femseg_seg','n_morphemes','correct','gold_seg','comment'])
    writer.writeheader(); writer.writerows(rows)

print(f'\n✅ CSV готов: {csv_path}')
print(f'   Слов для оценки: {len(rows)}')
print(f'   Коротких (4-5):  {len(short_w)}')
print(f'   Средних (6-9):   {len(mid_w)}')
print(f'   Длинных (10+):   {len(long_w)}')
print(f'\n   Отправьте файл эксперту-лингвисту по туркменскому языку.')
print(f'   Инструкция в шапке ноутбука.')


## 📥 Ячейка H3 — Инструкция для эксперта

Отправьте эксперту файл `human_eval_words.csv` со следующей инструкцией:

---

**Инструкция по заполнению Human Evaluation таблицы**

Вы оцениваете качество автоматической морфологической сегментации туркменских слов.
Система разделяет слово на морфемы через `@@`.

**Заполните колонку `correct`:**
- `TRUE` — сегментация полностью верна
- `FALSE` — сегментация неверна
- `PARTIAL` — частично верна (некоторые границы правильные)

**Если `FALSE` или `PARTIAL` — заполните `gold_seg`:**
Запишите правильную сегментацию через `@@ `

**Примеры:**
```
kitaplar  → kitap@@ lar        TRUE   (книги = корень + мн.ч.)
geldim    → gel@@ dim          TRUE   (пришёл = корень + прош.вр.)
häzirki   → h@@ äzirki         FALSE  häzirki  (целое слово, нельзя дробить)
adamlaryň → adam@@ lar@@ yň    TRUE
```

---
## 📊 Ячейка H4 — Подсчёт метрик (после получения ответа от эксперта)

Загрузите заполненный CSV и запустите эту ячейку.

In [ ]:
csv_filled = os.path.join(MODEL_BASE, 'human_eval_words_filled.csv')

if not os.path.exists(csv_filled):
    print(f'⚠️ Файл не найден: {csv_filled}')
else:
    with open(csv_filled, encoding='utf-8-sig') as f:
        reader = csv.reader(f, delimiter=';')
        rows = list(reader)

    data = []

    for row in rows:
        while len(row) < 6:
            row.append('')

        data.append({
            'word': row[0].strip(),
            'prediction': row[1].strip(),
            'count': row[2].strip(),
            'correct': row[3].strip(),
            'comment': row[4].strip(),
            'extra': row[5].strip(),
        })

    total = len(data)

    filled = [
        r for r in data
        if r['correct'].upper() in ('TRUE', 'FALSE', 'PARTIAL')
    ]

    true_count = sum(1 for r in filled if r['correct'].upper() == 'TRUE')
    false_count = sum(1 for r in filled if r['correct'].upper() == 'FALSE')
    partial_count = sum(1 for r in filled if r['correct'].upper() == 'PARTIAL')

    precision = true_count / max(1, len(filled))
    weighted = (true_count + 0.5 * partial_count) / max(1, len(filled))

    print('=' * 55)
    print('  HUMAN EVALUATION RESULTS')
    print('=' * 55)
    print(f'  Оценено слов:            {len(filled)} / {total}')
    print(f'  TRUE:                    {true_count}')
    print(f'  PARTIAL:                 {partial_count}')
    print(f'  FALSE:                   {false_count}')
    print('-' * 55)
    print(f'  Точность strict:         {precision*100:.1f}%')
    print(f'  Точность weighted:       {weighted*100:.1f}%')
    print('=' * 55)

    print('\nПо длине слова:')
    for label, min_l, max_l in [
        ('Короткие (4-5)', 4, 5),
        ('Средние (6-9)', 6, 9),
        ('Длинные (10+)', 10, 999),
    ]:
        group = [
            r for r in filled
            if min_l <= len(r['word']) <= max_l
        ]

        if not group:
            continue

        g_true = sum(1 for r in group if r['correct'].upper() == 'TRUE')

        print(f'  {label}: {g_true}/{len(group)} = {g_true/len(group)*100:.1f}%')

    human_results = {
        'total_evaluated': len(filled),
        'true': true_count,
        'partial': partial_count,
        'false': false_count,
        'accuracy_strict': round(precision * 100, 1),
        'accuracy_weighted': round(weighted * 100, 1),
    }

    with open(
        os.path.join(MODEL_BASE, 'human_eval_results.json'),
        'w',
        encoding='utf-8'
    ) as f:
        json.dump(human_results, f, ensure_ascii=False, indent=2)

    print('\n✅ Результаты сохранены')

  HUMAN EVALUATION RESULTS
  Оценено слов:            222 / 243
  TRUE:                    192
  PARTIAL:                 5
  FALSE:                   25
-------------------------------------------------------
  Точность strict:         86.5%
  Точность weighted:       87.6%

По длине слова:
  Короткие (4-5): 40/49 = 81.6%
  Средние (6-9): 86/97 = 88.7%
  Длинные (10+): 39/49 = 79.6%

✅ Результаты сохранены
